In [1]:
from chembl_webresource_client.new_client import new_client

# ChEMBL ka molecule database access
molecule = new_client.molecule

# Aspirin search karo
aspirin = molecule.filter(molecule_chembl_id='CHEMBL25')

# Result dekho
for m in aspirin:
    print("Name:", m['pref_name'])
    print("ChEMBL ID:", m['molecule_chembl_id'])
    print("SMILES:", m['molecule_structures']['canonical_smiles'])
    print("Max Phase:", m['max_phase'])

Name: ASPIRIN
ChEMBL ID: CHEMBL25
SMILES: CC(=O)Oc1ccccc1C(=O)O
Max Phase: 4.0


In [2]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule

# Apne UDS1b ka SMILES daalo
my_smiles = 'C1=CC=CC=C1C(C[Se]C(C2=CC=CC=C2)=O)=O'

# SMILES se similar molecules dhoondho
results = molecule.filter(
    molecule_structures__canonical_smiles__flexmatch=my_smiles
)

# First 5 results dekho
for m in list(results)[:5]:
    print("ChEMBL ID:", m['molecule_chembl_id'])
    print("Name:", m['pref_name'])
    print("SMILES:", m['molecule_structures']['canonical_smiles'])
    print("---")

In [3]:
from chembl_webresource_client.new_client import new_client

molecule = new_client.molecule

# Aspirin SMILES
aspirin_smiles = 'CC(=O)Oc1ccccc1C(=O)O'

# Search karo
results = molecule.filter(
    molecule_structures__canonical_smiles__flexmatch=aspirin_smiles
)

# Results dekho
count = 0
for m in results:
    if count < 5:
        print("ChEMBL ID:", m['molecule_chembl_id'])
        print("Name:", m['pref_name'])
        print("---")
        count += 1

ChEMBL ID: CHEMBL25
Name: ASPIRIN
---
ChEMBL ID: CHEMBL1697753
Name: ASPIRIN DL-LYSINE
---
ChEMBL ID: CHEMBL2296002
Name: None
---


In [4]:
from chembl_webresource_client.new_client import new_client

# Target database access
target = new_client.target

# GPX4 search karo
gpx4_targets = target.filter(pref_name__icontains='glutathione peroxidase 4')

# Results dekho
for t in gpx4_targets:
    print("ChEMBL ID:", t['target_chembl_id'])
    print("Name:", t['pref_name'])
    print("Organism:", t['organism'])
    print("Target Type:", t['target_type'])
    print("---")

In [5]:
from chembl_webresource_client.new_client import new_client

target = new_client.target

# Sirf GPX4 search karo (short name)
gpx4_targets = target.filter(pref_name__icontains='GPX4')

# Convert to list pehle (force fetch)
results = list(gpx4_targets)

print(f"Total results found: {len(results)}")
print("---")

for t in results[:5]:
    print("ChEMBL ID:", t['target_chembl_id'])
    print("Name:", t['pref_name'])
    print("Organism:", t['organism'])
    print("---")

Total results found: 2
---
ChEMBL ID: CHEMBL4295754
Name: Phospholipid hydroperoxide glutathione peroxidase GPX4
Organism: Homo sapiens
---
ChEMBL ID: CHEMBL4523148
Name: Phospholipid hydroperoxide glutathione peroxidase GPX4
Organism: Mus musculus
---


In [6]:
from chembl_webresource_client.new_client import new_client

# Activity database access
activity = new_client.activity

# Human GPX4 ke against tested molecules
gpx4_human = 'CHEMBL4295754'

# Bioactivity data lao
activities = activity.filter(
    target_chembl_id=gpx4_human,
    standard_type='IC50'
).only(['molecule_chembl_id', 'canonical_smiles', 
        'standard_value', 'standard_units'])

# List banao
results = list(activities)
print(f"Total bioactivity records: {len(results)}")
print("---")

# Pehle 5 dekho
for r in results[:5]:
    print("Molecule:", r['molecule_chembl_id'])
    print("IC50:", r['standard_value'], r['standard_units'])
    print("SMILES:", r['canonical_smiles'][:50], "...")
    print("---")

Total bioactivity records: 11
---
Molecule: CHEMBL5422297
IC50: 120.0 nM
SMILES: COc1cc(N(C(=O)CCl)c2ccc3c(c2)OCCO3)cc(OC)c1F ...
---
Molecule: CHEMBL5415644
IC50: 130.0 nM
SMILES: COc1cc(N(C(=O)CBr)c2ccc3c(c2)OCCO3)cc(OC)c1F ...
---
Molecule: CHEMBL4747331
IC50: 8420.0 nM
SMILES: COC(=O)c1ccc([C@H]2c3[nH]c4ccccc4c3C[C@H](C(=O)OC) ...
---
Molecule: CHEMBL1499544
IC50: 4850.0 nM
SMILES: COc1ccc(N(C(=O)CCl)C(C(=O)NCCc2ccccc2)c2cccs2)cc1C ...
---
Molecule: CHEMBL5570241
IC50: 542.5 nM
SMILES: COc1ccc2c(c1)NC(=O)/C2=C1\Nc2ccccc2\C1=N/OCCCNC(=O ...
---


In [7]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

activity = new_client.activity

# Human GPX4 ke against sab activities
activities = activity.filter(
    target_chembl_id='CHEMBL4295754',
    standard_type='IC50'
).only(['molecule_chembl_id', 'canonical_smiles', 
        'standard_value', 'standard_units'])

# DataFrame banao
df = pd.DataFrame(list(activities))

# Numeric conversion
df['IC50_nM'] = pd.to_numeric(df['standard_value'], errors='coerce')

# Sort karo best to worst
df_sorted = df.sort_values('IC50_nM')

# Display karo
print("GPX4 Inhibitors Ranked by IC50:")
print(df_sorted[['molecule_chembl_id', 'IC50_nM', 'standard_units']])

# CSV mein save karo
df_sorted.to_csv('gpx4_inhibitors.csv', index=False)
print("\n✅ Data saved to gpx4_inhibitors.csv")

GPX4 Inhibitors Ranked by IC50:
   molecule_chembl_id  IC50_nM standard_units
0       CHEMBL5422297    120.0             nM
1       CHEMBL5415644    130.0             nM
4       CHEMBL5570241    542.5             nM
6       CHEMBL1499544   2700.0             nM
3       CHEMBL1499544   4850.0             nM
9       CHEMBL5647419   6480.0             nM
2       CHEMBL4747331   8420.0             nM
5       CHEMBL5597052  10000.0             nM
10      CHEMBL1232461  10000.0             nM
7       CHEMBL5596393  26700.0             nM
8       CHEMBL5272075  86600.0             nM

✅ Data saved to gpx4_inhibitors.csv


In [8]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# Fingerprint generator
mfpgen = GetMorganGenerator(radius=2)

# Tumhara UDS1b
uds1b_smiles = 'C1=CC=CC=C1C(C[Se]C(C2=CC=CC=C2)=O)=O'
uds1b_mol = Chem.MolFromSmiles(uds1b_smiles)
uds1b_fp = mfpgen.GetFingerprint(uds1b_mol)

# CSV se data load karo
df = pd.read_csv('gpx4_inhibitors.csv')

# Similarity calculate karo har inhibitor se
similarities = []
for index, row in df.iterrows():
    mol = Chem.MolFromSmiles(row['canonical_smiles'])
    if mol:
        fp = mfpgen.GetFingerprint(mol)
        sim = DataStructs.TanimotoSimilarity(uds1b_fp, fp)
        similarities.append(round(sim * 100, 2))
    else:
        similarities.append(0)

# Column add karo
df['Similarity_to_UDS1b_%'] = similarities

# Sort by similarity
df_sorted = df.sort_values('Similarity_to_UDS1b_%', ascending=False)

# Display
print("🎯 GPX4 Inhibitors ranked by similarity to UDS1b:\n")
print(df_sorted[['molecule_chembl_id', 'IC50_nM', 'Similarity_to_UDS1b_%']])

🎯 GPX4 Inhibitors ranked by similarity to UDS1b:

   molecule_chembl_id  IC50_nM  Similarity_to_UDS1b_%
3       CHEMBL1499544   2700.0                  15.71
4       CHEMBL1499544   4850.0                  15.71
6       CHEMBL4747331   8420.0                  14.71
9       CHEMBL5596393  26700.0                  14.08
5       CHEMBL5647419   6480.0                  13.89
0       CHEMBL5422297    120.0                  11.67
1       CHEMBL5415644    130.0                  11.67
10      CHEMBL5272075  86600.0                  11.11
8       CHEMBL1232461  10000.0                   9.33
7       CHEMBL5597052  10000.0                   9.02
2       CHEMBL5570241    542.5                   8.59


In [ ]:
import matplotlib.pyplot as plt

# Sort by similarity
df_plot = df_sorted.reset_index(drop=True)

# Bar plot banao
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(df_plot)), 
              df_plot['Similarity_to_UDS1b_%'],
              color='steelblue', edgecolor='black')

# Highest similarity ko highlight karo
bars[0].set_color('red')

ax.set_xlabel('GPX4 Inhibitor Index', fontsize=12)
ax.set_ylabel('Similarity to UDS1b (%)', fontsize=12)
ax.set_title('UDS1b vs Known GPX4 Inhibitors: Structural Similarity', 
             fontsize=13, fontweight='bold')
ax.axhline(y=30, color='green', linestyle='--', label='Similarity threshold (30%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('uds1b_similarity_analysis.png', dpi=300)
plt.show()

print("✅ Figure saved as uds1b_similarity_analysis.png")

In [ ]:
import matplotlib.pyplot as plt

# Sort by similarity
df_plot = df_sorted.reset_index(drop=True)

# Bar plot banao
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(df_plot)), 
              df_plot['Similarity_to_UDS1b_%'],
              color='steelblue', edgecolor='black')

# Highest similarity ko highlight karo
bars[0].set_color('red')

ax.set_xlabel('GPX4 Inhibitor Index', fontsize=12)
ax.set_ylabel('Similarity to UDS1b (%)', fontsize=12)
ax.set_title('UDS1b vs Known GPX4 Inhibitors: Structural Similarity', 
             fontsize=13, fontweight='bold')
ax.axhline(y=30, color='green', linestyle='--', label='Similarity threshold (30%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('uds1b_similarity_analysis.png', dpi=300)
plt.show()

print("✅ Figure saved as uds1b_similarity_analysis.png")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# Fingerprint generator
mfpgen = GetMorganGenerator(radius=2)

# UDS1b
uds1b_smiles = 'C1=CC=CC=C1C(C[Se]C(C2=CC=CC=C2)=O)=O'
uds1b_mol = Chem.MolFromSmiles(uds1b_smiles)
uds1b_fp = mfpgen.GetFingerprint(uds1b_mol)

# CSV load karo
df = pd.read_csv('gpx4_inhibitors.csv')

# Similarity calculate
similarities = []
for index, row in df.iterrows():
    mol = Chem.MolFromSmiles(row['canonical_smiles'])
    if mol:
        fp = mfpgen.GetFingerprint(mol)
        sim = DataStructs.TanimotoSimilarity(uds1b_fp, fp)
        similarities.append(round(sim * 100, 2))
    else:
        similarities.append(0)

df['Similarity_to_UDS1b_%'] = similarities
df_sorted = df.sort_values('Similarity_to_UDS1b_%', ascending=False).reset_index(drop=True)

# Bar plot
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(df_sorted)), 
              df_sorted['Similarity_to_UDS1b_%'],
              color='steelblue', edgecolor='black')

bars[0].set_color('red')

ax.set_xlabel('GPX4 Inhibitor Index', fontsize=12)
ax.set_ylabel('Similarity to UDS1b (%)', fontsize=12)
ax.set_title('UDS1b vs Known GPX4 Inhibitors', fontsize=13, fontweight='bold')
ax.axhline(y=30, color='green', linestyle='--', label='Similarity threshold (30%)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('uds1b_similarity_analysis.png', dpi=300)
plt.show()

print("✅ Figure saved!")